# TSTR WGANGP Dataset A - Diabetes

In [1]:
#import libraries
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import os
print('Libraries imported!!')

Libraries imported!!


In [2]:
#define directory of functions and actual directory
FUNCTIONS_HOME = '../../../functions/evaluation_functions/' #home directory of the project
REAL_DATA_HOME = '../../../data/raw/chap/' #home directory of the project
SYN_DATA_HOME  = '../../../data/processed/chap/' #home directory of the project
FUNCTIONS_DIR = 'EVALUATION FUNCTIONS/UTILITY'
ACTUAL_DIR = os.getcwd()

#change directory to functions directory
os.chdir(FUNCTIONS_HOME + FUNCTIONS_DIR)

#import functions for data labelling analisys
from utility_evaluation import DataPreProcessor
from utility_evaluation import train_evaluate_model

#change directory to actual directory
os.chdir(ACTUAL_DIR)
print('Functions imported!!')

Functions imported!!


## 1. Read data

In [3]:
#read real dataset
train_data = pd.read_csv(SYN_DATA_HOME + '1_Chap_Data_Synthetic_WGANGP.csv')
categorical_columns = ['group']
for col in categorical_columns :
    train_data[col] = train_data[col].astype('category')
train_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,-8.541844,-0.545287,-0.928663,-0.455110,-4.351696,-0.234827,-0.769859,-0.550513
1,Group0,-6.892333,-0.171208,-1.117848,-0.472040,-2.482052,-0.165285,-0.761114,-0.493305
2,Group0,-8.609503,-0.617723,-0.825237,-0.471001,-4.817324,-0.072427,-0.707525,-0.523854
3,Group0,-6.244475,-0.107295,-1.029050,-0.459597,-2.165540,0.000130,-0.753020,-0.461165
4,Group0,-4.694835,0.177444,-0.970630,-0.408257,-0.376301,0.362118,-0.596771,-0.342734
...,...,...,...,...,...,...,...,...,...
2712,Group0,-5.952104,-0.110032,-0.914333,-0.425598,-1.995470,0.118183,-0.695510,-0.433784
2713,Group0,-7.202361,-0.255103,-1.023678,-0.456831,-2.909712,-0.190980,-0.777772,-0.505518
2714,Group0,-5.041869,0.095305,-0.988946,-0.426792,-0.824161,0.251926,-0.629764,-0.374986
2715,Group0,-6.180864,-0.117613,-0.936792,-0.430850,-2.062372,0.121498,-0.726012,-0.449142


In [4]:
#read test data
test_data = pd.read_csv(REAL_DATA_HOME + '1_Chap_Data_Real_Test.csv')
for col in categorical_columns :
    test_data[col] = test_data[col].astype('category')
test_data

,group,sbp1,sbp2,sbp3,sbp4,dbp1,dbp2,dbp3,dbp4
0,Group0,-3.053475,-0.177497,-0.010661,0.205118,-2.773102,-0.653440,0.319585,0.017457
1,Group0,-1.375254,0.761393,0.270692,0.140597,-2.157208,-0.297755,0.126143,0.118270
2,Group0,3.681521,1.147414,-0.119372,0.276912,1.452468,0.068735,0.043671,0.074436
3,Group0,0.383070,0.211771,0.374537,0.254718,5.352641,-0.611636,0.344336,0.044627
4,Group0,2.009235,0.765321,0.100557,0.058659,2.842860,0.980271,-0.014527,0.096506
...,...,...,...,...,...,...,...,...,...
674,Group0,1.185955,-0.248240,-0.258972,0.284599,0.816753,0.553623,-0.456024,0.157975
675,Group0,-4.898967,-0.576216,-0.150236,0.069086,-4.450551,-0.192307,0.034382,-0.174072
676,Group0,-3.339095,0.856460,-1.021265,-0.131611,0.639099,0.783467,-0.128578,-0.067689
677,Group0,-3.844234,0.083773,0.334898,-0.210940,-1.259774,0.726944,-0.299066,-0.123455


In [5]:
target = 'group'
#quick look at the breakdown of class values
print('Train data')
print(train_data.shape)
print(train_data.groupby(target).size())
print('#####################################')
print('Test data')
print(test_data.shape)
print(test_data.groupby(target).size())

Train data
(2717, 9)
group
Group0    2670
Group1      47
dtype: int64
#####################################
Test data
(679, 9)
group
Group0    645
Group1     34
dtype: int64


## 2. Pre-process training data

In [6]:
target = 'group'
categorical_columns = []
numerical_columns = train_data.select_dtypes(include=['int64','float64']).columns.tolist()
categories = [np.array(range(2))] if categorical_columns else []
data_preprocessor = DataPreProcessor(categorical_columns, numerical_columns, categories)
x_train = data_preprocessor.preprocess_train_data(train_data.loc[:, train_data.columns != target])
y_train = train_data.loc[:, target]

x_train.shape, y_train.shape

((2717, 8), (2717,))

## 3. Preprocess test data

In [7]:
x_test = data_preprocessor.preprocess_test_data(test_data.loc[:, test_data.columns != target])
y_test = test_data.loc[:, target]
x_test.shape, y_test.shape

((679, 8), (679,))

## 4. Create a dataset to save the results

In [8]:
results = pd.DataFrame(columns = ['model','accuracy','precision','recall','f1'])
results

,model,accuracy,precision,recall,f1


## 4. Train and evaluate Random Forest Classifier

In [9]:
rf_results = train_evaluate_model('RF', x_train, y_train, x_test, y_test)
results = pd.concat([results, rf_results], ignore_index=True)
rf_results

[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    0.0s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=3)]: Using backend ThreadingBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  44 tasks      | elapsed:    0.0s
[Parallel(n_jobs=3)]: Done 100 out of 100 | elapsed:    0.0s finished


,model,accuracy,precision,recall,f1
0,RF,0.8159,0.9067,0.8159,0.8571


## 5. Train and Evaluate KNeighbors Classifier

In [10]:
knn_results = train_evaluate_model('KNN', x_train, y_train, x_test, y_test)
results = pd.concat([results, knn_results], ignore_index=True)
knn_results

,model,accuracy,precision,recall,f1
0,KNN,0.947,0.9022,0.947,0.9241


## 6. Train and evaluate Decision Tree Classifier

In [11]:
dt_results = train_evaluate_model('DT', x_train, y_train, x_test, y_test)
results = pd.concat([results, dt_results], ignore_index=True)
dt_results

,model,accuracy,precision,recall,f1
0,DT,0.8115,0.9044,0.8115,0.8539


## 7. Train and evaluate Support Vector Machines Classifier

In [12]:
svm_results = train_evaluate_model('SVM', x_train, y_train, x_test, y_test)
svm_results['model'] = svm_results['model'].replace(['MLP'],'SVM')
results = pd.concat([results, svm_results], ignore_index=True)
svm_results

[LibSVM]WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -301.101964, rho = 50.200696
nSV = 9, nBSV = 2
Total nSV = 9
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -196.512525, rho = 53.123751
nSV = 7, nBSV = 1
Total nSV = 7
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -528.295347, rho = 42.981982
nSV = 11, nBSV = 3
Total nSV = 11
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -478.944004, rho = 46.403876
nSV = 7, nBSV = 2
Total nSV = 7
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -534.749846, rho = 37.458487
nSV = 9, nBSV = 3
Total nSV = 9
WARN: libsvm Solver reached max_iter
optimization finished, #iter = 300
obj = -541.420666, rho = -48.029152
nSV = 12, nBSV = 4
Total nSV = 12


,model,accuracy,precision,recall,f1
0,SVM,0.7968,0.9034,0.7968,0.845


## 8. Train and evaluate Multilayer Perceptron Classifier

In [13]:
mlp_results = train_evaluate_model('MLP', x_train, y_train, x_test, y_test)
results = pd.concat([results, mlp_results], ignore_index=True)
mlp_results

Iteration 1, loss = 0.55036108
Iteration 2, loss = 0.28039693
Iteration 3, loss = 0.13559707
Iteration 4, loss = 0.07729889
Iteration 5, loss = 0.04251972
Iteration 6, loss = 0.02601991
Iteration 7, loss = 0.01998120
Iteration 8, loss = 0.01810582
Iteration 9, loss = 0.01827761
Iteration 10, loss = 0.01444137
Iteration 11, loss = 0.01313778
Iteration 12, loss = 0.01225520
Iteration 13, loss = 0.01226368
Iteration 14, loss = 0.01123636
Iteration 15, loss = 0.01143596
Iteration 16, loss = 0.01094048
Iteration 17, loss = 0.01038280
Iteration 18, loss = 0.00992033
Iteration 19, loss = 0.00989649
Iteration 20, loss = 0.00950868
Iteration 21, loss = 0.00912627
Iteration 22, loss = 0.00913923
Iteration 23, loss = 0.00884516
Iteration 24, loss = 0.00995108
Iteration 25, loss = 0.00832005
Iteration 26, loss = 0.00807942
Iteration 27, loss = 0.00835764
Iteration 28, loss = 0.00890837
Iteration 29, loss = 0.00888034
Iteration 30, loss = 0.00820219
Iteration 31, loss = 0.00892192
Iteration 32, los

,model,accuracy,precision,recall,f1
0,MLP,0.9485,0.9023,0.9485,0.9248


## 9. Save results file

In [14]:
results.to_csv('RESULTS/models_results_wgangp.csv', index=False)
results

,model,accuracy,precision,recall,f1
0,RF,0.8159,0.9067,0.8159,0.8571
1,KNN,0.9470,0.9022,0.9470,0.9241
2,DT,0.8115,0.9044,0.8115,0.8539
3,SVM,0.7968,0.9034,0.7968,0.8450
4,MLP,0.9485,0.9023,0.9485,0.9248
